In [1]:
import numpy as np
import pandas as pd

from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder, TrainValidationSplit
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
#This transforms data
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import StandardScaler

import time

## Reading data from server

In [2]:
# dfPandas = pd.read_csv('data/diabetes_012.csv')

# dfPandas.head(5)

# Path to your downloaded sqlite-jdbc JAR

# jdbc_jar_path = "sqlite-jdbc-3.51.3.0.jar"

# spark = SparkSession.builder \
#     .appName("SQLite_Query") \
#     .config("spark.jars", jdbc_jar_path) \
#     .config("spark.driver.extraClassPath", jdbc_jar_path) \
#     .getOrCreate()


In [4]:
jdbc_jar_path = "sqlite-jdbc-3.51.3.0.jar"

builder = SparkSession.builder
builder = builder.config("spark.executor.memory", "2G")
builder = builder.config("spark.driver.memory", "10G")
builder = builder.config("spark.driver.maxResultSize", "5G")

builder = builder.config("spark.executor.instances", "3")
# builder = builder.config()
# builder = builder.config()
# builder = builder.config()
#SQL database connector
builder = builder.config("spark.jars", jdbc_jar_path)
builder = builder.config("spark.driver.extraClassPath", jdbc_jar_path)

spark = builder.appName("RandomForest").getOrCreate()
sc = spark.sparkContextsc = spark.sparkContext

In [26]:
# .config("spark.executor.instances", "4") \
#     .config("spark.executor.cores", "2") \
#     .config("spark.executor.memory", "4g") \

# from pyspark.sql import SparkSession
# spark = SparkSession.builder \
#     .config("spark.jars.packages", "sqlite-jdbc-3.34.0.jar") \
#     .getOrCreate()
# # .config("spark.jars.packages", "org.xerial:sqlite-jdbc:3.45.2.0") \

In [27]:
#Activate Spark
# spark = SparkSession.builder.appName("RandomForest").config("spark.executor.instances", "4").getOrCreate()
# spark = SparkSession.builder.appName("RandomForest").getOrCreate()

# spark = SparkSession.builder.appName("RandomForest").config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.minExecutors", "1").config("spark.dynamicAllocation.maxExecutors", "3").config("spark.executor.memory", "4g").getOrCreate()

# spark.conf.set("spark.dynamicAllocation.enabled", "true")
# spark.conf.set("spark.dynamicAllocation.minExecutors", "1")
# spark.conf.set("spark.dynamicAllocation.maxExecutors", "10")
# # Requires external shuffle service to be enabled in most environments
# spark.conf.set("spark.shuffle.service.enabled", "true")

In [5]:
db_url = "jdbc:sqlite:data/database.db"
table_name = "diabetes_012"

# Load the table into a DataFrame
df = spark.read \
    .format("jdbc") \
    .option("url", db_url) \
    .option("dbtable", table_name) \
    .option("driver", "org.sqlite.JDBC") \
    .load()

# Display results
df.show(5)

+-----+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|label|HighBP|HighChol|CholCheck| BMI|Smoker|Stroke|HeartDiseaseorAttack|PhysActivity|Fruits|Veggies|HvyAlcoholConsump|AnyHealthcare|NoDocbcCost|GenHlth|MentHlth|PhysHlth|DiffWalk|Sex| Age|Education|Income|
+-----+------+--------+---------+----+------+------+--------------------+------------+------+-------+-----------------+-------------+-----------+-------+--------+--------+--------+---+----+---------+------+
|  0.0|   1.0|     1.0|      1.0|40.0|   1.0|   0.0|                 0.0|         0.0|   0.0|    1.0|              0.0|          1.0|        0.0|    5.0|    18.0|    15.0|     1.0|0.0| 9.0|      4.0|   3.0|
|  0.0|   0.0|     0.0|      0.0|25.0|   1.0|   0.0|                 0.0|         1.0|   0.0|    0.0|              0.0|          0.0|        1.0|    3.0|     0.0|     0.0| 

In [6]:
# spark.stop()

In [7]:
# df = spark.read.csv('data/diabetes_012.csv',inferSchema=True, header=True)

In [8]:
# df = df.sample(withReplacement=False, fraction=0.3, seed=42)

In [9]:
# df.show(5)

In [10]:
inputColumns=["HighBP", "HighChol", "CholCheck","BMI","Smoker","Stroke","HeartDiseaseorAttack","PhysActivity","Fruits","Veggies","HvyAlcoholConsump","AnyHealthcare","NoDocbcCost","GenHlth","MentHlth","PhysHlth","DiffWalk","Sex", "Age","Education","Income"]

In [11]:
assembler = VectorAssembler(
    inputCols=inputColumns,
    outputCol="features")
vectorDF = assembler.transform(df)

In [12]:
vectorDF = vectorDF.drop(*inputColumns)
vectorDF.show(5)

26/04/14 06:46:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-----+--------------------+
|label|            features|
+-----+--------------------+
|  0.0|[1.0,1.0,1.0,40.0...|
|  0.0|(21,[3,4,7,12,13,...|
|  0.0|[1.0,1.0,1.0,28.0...|
|  0.0|(21,[0,2,3,7,8,9,...|
|  0.0|[1.0,1.0,1.0,24.0...|
+-----+--------------------+
only showing top 5 rows


In [13]:
scaler = StandardScaler(
    inputCol="features", 
    outputCol="scaledFeatures",
    withStd=True, 
    withMean=True
)

scaler_model = scaler.fit(vectorDF)

vectorDF = scaler_model.transform(vectorDF).drop("features")
# vectorDF = vectorDF.cache()et checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext.'), Param(parent='RandomForestClassifier_ba4552808cd7', name='featureSubsetStrategy', doc="The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees == 1, set to 'all'. If numTrees > 1 (forest), set to 'sqrt' for classification and to 'onethird' for regression), 'all' (use all features), 'onethird' (use 1/3 of the features), 'sqrt' (use sqrt(number of features)), 'log2' (use log2(number of features)), 'n' (when n is in the range (0, 1.0], use n * number of features. When n is in the range (1, number of features), use n features). default = 'auto'"), Param(parent='RandomForestClassifier_ba4552808cd7', name='featuresCol', doc='features column name.'), Param(parent='RandomForestClassifier_ba4552808cd7', name='impurity', doc='Criterion used for information gain calculation (case-insensitive). Supported options: entropy, gini'
# broadcastDF = sc.broadcast(vectorDF)

In [14]:
vectorDF.show(5)

+-----+--------------------+
|label|      scaledFeatures|
+-----+--------------------+
|  0.0|[1.15368587013306...|
|  0.0|[-0.8667836574183...|
|  0.0|[1.15368587013306...|
|  0.0|[1.15368587013306...|
|  0.0|[1.15368587013306...|
+-----+--------------------+
only showing top 5 rows


In [15]:
numExecutors = 3
vectorDF = vectorDF.repartition(numExecutors)
# df = df.repartition("customer_id")

In [16]:
forest = RandomForestClassifier(featuresCol="scaledFeatures", labelCol="label", seed=42)

In [17]:
# paramGrid = (ParamGridBuilder()
#              .addGrid(forest.numTrees, [100, 200, 300])
#              .addGrid(forest.maxDepth, [5, 10, 15, 20])
#              .build())

paramGrid = (ParamGridBuilder()
             .addGrid(forest.numTrees, [100, 200])
             .addGrid(forest.maxDepth, [10, 15, 20])
             .addGrid(forest.maxBins, [32, 40])
             .build())

evaluator = MulticlassClassificationEvaluator(metricName="accuracy")

# cv = CrossValidator(estimator=forest,
#                     estimatorParamMaps=paramGrid,
#                     evaluator=evaluator,
#                     numFolds=3)

cv = TrainValidationSplit(estimator=forest,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    trainRatio=0.8,
                    parallelism=numExecutors)

In [18]:

# Run grid search
startTime = time.perf_counter()
cvModel = cv.fit(vectorDF)

# Access the best model
bestModel = cvModel.bestModel

endTime = time.perf_counter()
print(f"Elapsed time: {endTime - startTime:.6f} seconds")

26/04/14 06:46:51 WARN BlockManager: Block rdd_25_1 already exists on this machine; not re-adding it
26/04/14 06:46:51 WARN BlockManager: Block rdd_25_1 already exists on this machine; not re-adding it
26/04/14 06:46:51 WARN BlockManager: Block rdd_25_0 already exists on this machine; not re-adding it
26/04/14 06:46:51 WARN BlockManager: Block rdd_25_0 already exists on this machine; not re-adding it
26/04/14 06:46:51 WARN BlockManager: Block rdd_25_2 already exists on this machine; not re-adding it
26/04/14 06:46:57 WARN DAGScheduler: Broadcasting large task binary with size 1030.8 KiB
26/04/14 06:46:57 WARN DAGScheduler: Broadcasting large task binary with size 1030.8 KiB
26/04/14 06:46:59 WARN DAGScheduler: Broadcasting large task binary with size 2003.9 KiB
26/04/14 06:46:59 WARN DAGScheduler: Broadcasting large task binary with size 2004.0 KiB
26/04/14 06:47:00 WARN DAGScheduler: Broadcasting large task binary with size 1030.8 KiB
26/04/14 06:47:01 WARN DAGScheduler: Broadcasting 

Elapsed time: 2082.955135 seconds


In [19]:
#204 secs vs 498 seconds with
#current: 2082 seconds
summary = bestModel.summary
# print(summary.objectiveHistory)
# print(f"RMSE: {summary.rootMeanSquaredError}")
# print(f"R2: {summary.r2}")
# summary.residuals.show() # Show residuals
# print(bestModel.params)

params = bestModel.extractParamMap()

# Print parameters without docstrings
for param, value in params.items():
    print(f"{param.name}: {value}",end=", ")
    
print('\naccuracy:', summary.accuracy)
print('precision:',summary.weightedPrecision)
print(summary.totalIterations)

bootstrap: True, cacheNodeIds: False, checkpointInterval: 10, featureSubsetStrategy: auto, featuresCol: scaledFeatures, impurity: gini, labelCol: label, leafCol: , maxBins: 40, maxDepth: 15, maxMemoryInMB: 256, minInfoGain: 0.0, minInstancesPerNode: 1, minWeightFractionPerNode: 0.0, numTrees: 200, predictionCol: prediction, probabilityCol: probability, rawPredictionCol: rawPrediction, seed: 42, subsamplingRate: 1.0, 

26/04/14 07:23:33 WARN DAGScheduler: Broadcasting large task binary with size 169.2 MiB
[Stage 645:======================================>                  (2 + 1) / 3]


accuracy: 0.8763481551561022
precision: 0.8738856171731628
0


In [42]:
spark.stop()

ConnectionRefusedError: [Errno 111] Connection refused

In [19]:
spark.sparkContext.cancelAllJobs()

26/04/14 06:13:02 ERROR Instrumentation: org.apache.spark.SparkException: [SPARK_JOB_CANCELLED] Job 249 cancelled as part of cancellation of all jobs SQLSTATE: XXKDA
	at org.apache.spark.errors.SparkCoreErrors$.sparkJobCancelled(SparkCoreErrors.scala:222)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:3037)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$doCancelAllJobs$2(DAGScheduler.scala:1151)
	at scala.runtime.java8.JFunction1$mcVI$sp.apply(JFunction1$mcVI$sp.scala:18)
	at scala.collection.mutable.HashSet$Node.foreach(HashSet.scala:450)
	at scala.collection.mutable.HashSet.foreach(HashSet.scala:376)
	at org.apache.spark.scheduler.DAGScheduler.doCancelAllJobs(DAGScheduler.scala:1150)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.doOnReceive(DAGScheduler.scala:3359)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive(DAGScheduler.scala:3328)
	at org.apache.spark.scheduler.DAGSchedulerEventProcessLoop.onReceive